# FilSeg Gap Research — Optional Colab A100 Accelerator
Runs existing `gap_research/kernels/{exp008a_frcnn5f,exp008b_convnext_topo}` unmodified on Colab A100 (40/80GB) by staging Kaggle inputs under `/kaggle/input` and uploading compact `.tar.gz` results to private Kaggle dataset `ajikajadeh/filseg-gap-a100-out`.
**Zero credentials are embedded**: reads Colab Secrets (`KAGGLE_USERNAME`/`KAGGLE_KEY`/`KAGGLE_API_TOKEN`) or existing `~/.kaggle/kaggle.json`.

In [ ]:
# @title 1. Configure Task & Verify A100 GPU + Kaggle Auth
TASK = "exp008b_convnext_topo"  # @param ["exp008a_frcnn5f", "exp008b_convnext_topo"]
REPO_BRANCH = "gap-research/0.41-plus"

import os, subprocess, pathlib
try:
    from google.colab import userdata
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"):
        try:
            v = userdata.get(k)
            if v:
                os.environ[k] = v
        except Exception:
            pass
except ImportError:
    pass

!nvidia-smi -L
!pip install -q kaggle ultralytics==8.4.160 segmentation-models-pytorch==0.5.0 pycocotools scikit-image 'transformers==5.14.1'


In [ ]:
# @title 2. Clone Isolated Gap Branch & Stage Read-Only Kaggle Inputs
import os, subprocess, pathlib, json

repo = pathlib.Path("/content/filseg-gap")
if not repo.exists():
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, "https://github.com/yauwes/solar-filament-segmentation-2026.git", str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)

k_in = pathlib.Path("/kaggle/input")
k_work = pathlib.Path("/kaggle/working")
k_in.mkdir(parents=True, exist_ok=True)
k_work.mkdir(parents=True, exist_ok=True)

# Download competition data & weights once
comp_dir = k_in / "competitions" / "filament-segmentation-2026"
if not comp_dir.exists():
    comp_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["kaggle", "competitions", "download", "-c", "filament-segmentation-2026", "-p", str(comp_dir)], check=True)
    for z in comp_dir.glob("*.zip"):
        subprocess.run(["unzip", "-qo", str(z), "-d", str(comp_dir)], check=True)
        z.unlink()

meta_path = repo / "gap_research" / "kernels" / TASK / "kernel-metadata.json"
meta = json.loads(meta_path.read_text())
for ds in meta.get("dataset_sources", []):
    slug = ds.split("/")[-1]
    dst = k_in / "datasets" / ds
    if not dst.exists():
        dst.mkdir(parents=True, exist_ok=True)
        subprocess.run(["kaggle", "datasets", "download", "-d", ds, "-p", str(dst), "--unzip"], check=True)
for ks in meta.get("kernel_sources", []):
    dst = k_in / "notebooks" / ks
    if not dst.exists():
        dst.mkdir(parents=True, exist_ok=True)
        subprocess.run(["kaggle", "kernels", "output", ks, "-p", str(dst)], check=True)
print("All Kaggle inputs staged under /kaggle/input")


In [ ]:
# @title 3. Execute Kernel on A100 & Upload Bundles to ajikajadeh/filseg-gap-a100-out
import pathlib, shutil, json, subprocess, torch

script = repo / "gap_research" / "kernels" / TASK / meta["code_file"]
env = dict(os.environ)
if torch.cuda.device_count() == 1:
    # Map any CUDA_VISIBLE_DEVICES="1" child spawn safely to single A100 when running 1-GPU runtime
    patched = pathlib.Path("/tmp/colab_run_script.py")
    patched.write_text(script.read_text().replace('CUDA_VISIBLE_DEVICES=str(g)', 'CUDA_VISIBLE_DEVICES="0"').replace('CUDA_VISIBLE_DEVICES="1"', 'CUDA_VISIBLE_DEVICES="0"'))
    script = patched

subprocess.run(["python3", str(script)], check=True)

# Push output archives (skipping heavy weights if desired, or keeping compact bundles) to Kaggle dataset
out_pkg = pathlib.Path("/tmp/a100_out_upload")
out_pkg.mkdir(parents=True, exist_ok=True)
for f in k_work.glob(f"{TASK[:7]}*"):
    if f.is_file() and "weights" not in f.name:
        shutil.copy2(f, out_pkg / f.name)
(out_pkg / "dataset-metadata.json").write_text(json.dumps({
    "title": "filseg gap a100 out",
    "id": "ajikajadeh/filseg-gap-a100-out",
    "licenses": [{"name": "CC0-1.0"}]
}, indent=2))
# Create if new, else version
rc = subprocess.call(["kaggle", "datasets", "version", "-p", str(out_pkg), "-m", f"A100 {TASK} outputs"])
if rc != 0:
    subprocess.run(["kaggle", "datasets", "create", "-p", str(out_pkg)], check=True)
